# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# Write your solution here
from pyspark.sql.functions import *

# Extract
bookings_df = spark.table("practice.bronze_bookings")

# Transform (Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.)
filtered_bookings_df = (
    bookings_df.filter((col("starttime") >= "2012-09-01") & (col("starttime") < "2012-10-01"))
    .groupBy("facid")
    .agg(sum("slots").alias("total_slots"))
    .orderBy(col("total_slots"))
)

# Load
filtered_bookings_df.write.mode("overwrite").saveAsTable("practice.slots_per_facility_september2012")

display(filtered_bookings_df)


facid,total_slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here

# Extract
bookings_df = spark.table("practice.bronze_bookings")
facilities_df = spark.table("practice.bronze_facilities")
members_df = spark.table("practice.bronze_members")

# Transform (How can you produce a list of all members who have used a tennis court? )
joined_df = (
    members_df.alias("m")
    .join(bookings_df.alias("b"), col("m.memid") == col("b.memid"))
    .join(facilities_df.alias("f"), col("b.facid") == col("f.facid"))
    .filter(col("f.name").like("Tennis Court%"))
    .select(
        concat_ws(" ", col("m.surname"), col("m.firstname")).alias("member"), 
        col("f.name").alias("facility")
    )
    .distinct()
    .orderBy(col("member"), col("facility"))
)

# Load
joined_df.write.mode("overwrite").format("delta").partitionBy("facility").saveAsTable("practice.threejoin_delta")

display(joined_df)



member,facility
Bader Florence,Tennis Court 1
Bader Florence,Tennis Court 2
Baker Anne,Tennis Court 1
Baker Anne,Tennis Court 2
Baker Timothy,Tennis Court 1
Baker Timothy,Tennis Court 2
Boothe Tim,Tennis Court 1
Boothe Tim,Tennis Court 2
Butters Gerald,Tennis Court 1
Butters Gerald,Tennis Court 2


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Write your solution here
import requests
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Extract (HTTP requests)
# Company symbols
companies = {
    "Google": "GOOGL",
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Tesla": "TSLA"
}

# API configuration
url = "https://alpha-vantage.p.rapidapi.com/query"

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

# Schema for Spark DataFrame
schema = StructType([
    StructField("company", StringType(), True),
    StructField("date", DateType(), True),
    StructField("close", DoubleType(), True)
])

# Extract
final_df = None

for company, symbol in companies.items():
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    time_series = data.get("Time Series (Daily)", {})

    rows = []

    for date, values in time_series.items():
        close_price = float(values["4. close"])
        row = (company, date, close_price)
        rows.append(row)


    spark_df = spark.createDataFrame(rows, schema=schema)

    if final_df is None:
        final_df = spark_df
    else:
        final_df = final_df.union(spark_df)

# Transform (Weekly max closing price)
weekly_df = (
    final_df
    .withColumn("year", year(col("date")))
    .withColumn("week", weekofyear(col("date")))
)

weekly_max_df = (
    weekly_df
    .groupBy("company", "year", "week")
    .agg(max("close").alias("max_closing_price"))
    .orderBy("company", "year", "week")
)

# Load
weekly_max_df.write.mode("overwrite").partitionBy("company").saveAsTable("max_closing_price_weekly")


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Write your solution here

# Extract
# jdbc:<database_type>://<host>:<port>/<database_name>
jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"

query = "(SELECT * FROM rna LIMIT 100) AS rna_subquery"

rna_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", query)
    .option("user", "reader")
    .option("password", "NWDMCE5xdipIjRrp")
    .option("driver", "org.postgresql.Driver")
    .load()
)

# No transformation

# Load
rna_df.write.mode("overwrite").saveAsTable("rna_100_records")

display(rna_df)


id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
15642362,URS0000EEAEFA,2019-08-29T05:46:56.644Z,rnacen,0A262848178EAB6E,1603,CCACCAGAACTATCAGGGTGATACACTTTCATCAGGGGGCTCAGGCTGGCGAGATTCACCAGATCCAGGATGGGCAGCTTCAAGACAGGGCTCCACACCATTATTTGCGGACCTCTGGAGCGTTCCTCAAAACCACTGATGGTCCTTCAAACCTCTCATTTCCAGGCGAGAACAGTACAGTCTGCCTGGTGCTGAAGCCAGAGCTCAGGCTTTGGCTGATGCTGAGAGCAAACCTCTGCTGTACAGGCATATACTTTGAATCACCGCTCCTATGAAATTCATATTAATCTGTTCATCTCTGAATCAGACACTTTCTTCCCCTCTCTCCAGTACACTACAAGTGGAAACATCTTGACAATCACTGAACAAGCCATGCGCTTCCTTCCTCCCGGCCTCTCACACAGTATGGGCCCTCTATCTGGAATCCTCTTCCACTTTTCCTCTGCCTGGGAAACTCCTTCCCACTCTTCAGATGCAGCTCAAAAGTGACCTTCTCCATCAAGGTTTTCCTACCTCTTCCAGGCAGGACTGTTCCGCCCTCTGCACACCCACATCCCTTAGGACAGGCATCTCCCACAAATTCTCACCTCGGCTTTGACTACGGGTTTAGGTGCTCATCTTTCCATGCGCCCCCTGGGGAGTGAGGTCTTATTTATCTCTGTACCAATTCTCATCCTTGCTTTGCCCCAGTGGCTAGTGCGGGGCTTGGCACACCACAGGCCTTCATTACAATATCATTTCCATGGAACTGTGACAGGAACTCCCACTAACCTTTCAAATGTTGGCTTTAGGTGTGAAACCACACATGAAATGTCAAGTGTATGTATGCACGTGTATGTGTTGACATATGGAATGCTCTTCACTAATTCAGACCTCTCTTTACCTCCTCAGGGTGAGATTATTCGGAGAAATAATAATGGCTGATGTTTGTATAACACCTTGAATGTATTTGCTTTCAGAGACATTATCTGATCTGTGGTACCGAGCAAAGAGCAAATCAATCAACCTGATGTATGTTGATGTTGACACGGATTTTGGAGTGGAAAAGTCTTTGCTGAGTTTGTGCCCTGATTACATGTCCGTGCCTGAAAACGCTGACCATAGGTCCCTGCTGTGAGAAGCAAGATCTGCAAAACTGGAACATACCTGATAACATGCGCATCCCTTACTATCCTCCATAAATAGATTTTAGCAAGTCAGAAAGTTTGAACAAAAACTCATAATTCTTGTCAGTAATAGTGGACACTCCCAATTAAGAAAACACTCACTTATTTTTTCCTACTGCCATCGAGGAAAGCTGGAAAAAAGCATTTGGCAAATCATCTTCTGTCATCCTTAGATTTCAGGAGGATAAGTTCAAACAGATGAAACAGAAGTTCTGCAGTAACTCATTCTACTGTACAATTCAACCCAACCAACTCCCGGGGCTCTTGAGCTAGCTGCCTGTCATCAAGGATGTTTGGATGAGTTCTATGAGCTGTCAGGGTTTAATAAGCCAAGCTAGTTAATGAGTGATGGCTAGATTTCATTCCAAGACTTCACATTATCCATGAAGCAATTAAGCATGCATCTAGTGGTCTCCCACGGGAGTTCCAAAGGTGCC,null,026def1ea7d08f2d5cbbd93136db3ae4
15653110,URS0000EED8F6,2019-08-29T05:46:56.645Z,rnacen,C79F3CEE83A24E42,133,AGCTTTGTGCAATGGCAGCCAATGAGGTTTACCTGAGGTGTGATTATTGCTAATGGAAAACTCTTCTCAGTACCCTGCCATGATGTCTTGAAACATAGTCAGCATCGGCAATTTTTGACAGTCTCTCCAGAGA,null,38a4fb44aff901fac3a4d9ae7c010aef
15672578,URS0000EF2502,2019-08-29T05:46:56.645Z,rnacen,C7A486AD0A6A7D69,754,ATGCGTCCCCAAAGTAGGTTGACTTTCTGGGACTGCTTAAAGCGATTCCTCCCGACAAAGGGAAGGGTAGTGCCTTCCTCAGTCTCTTGGGGCATACTTCTTTCCGGATCTGGCACAAACTACCCCGCAGGGCTTAAAGACATGCTGATGACAGATGATGAAGGTATGCATCTCTTAATCAAGCTGCCTCACTTGCCCATCTAAAGAGCTATTTAAAAAACACATCTAGCAGGTCTCAGCCGCCAATATTCACATCTTATTTACTCCTACAGTCACTTTTGAAAAATCTAAAGAAATTCTCATTCAGGCTATGACCTAAGGAGAGGGCAAGCTGAAAATTCCCTCTGCAGGGCAAGGGCATTTTTTTAGTTTAAATTCAGTTGCTGATGTTATTAGCTGCTGTTCTGACCTTCCAGCCAACTTAAAGATGGTTTATGTTAACATTGAAAATGTCAACTCGTGTGGCCAGAGTGTTTTCTAAATGATGCTCTTTACATGGTTCTTCATTTGTTGGATAATAATGACTATTTTCTCCTTTGGCTTTAAAACTGAAGGAATATGGCTAGGTTTCCTTCTCTGTTTGCAAATGGAGCCCCTGCCATGTGTCTAGGCCTGGGCTGGGTTCTTTCTACCCCATCCTATTCCTCTGCCAAGAAAGCAAAGATCATATTTCAAACATATTATTTCCACCTTGAACAAAGTTGGTGGGAATGTAAATTTTGTGCAGCCACTGTGGAAAAGTATAGAAGTTTCT,null,9babbe77c31291a62f511ae6cb2c88eb
15664783,URS0000EF068F,2019-08-29T05:46:56.646Z,rnacen,76868E3F4AA1C1EB,107,ATGCTGACTTTGGCAGCACATAGACTACAATTGGAATGATACAGAGAAGACTAGGATGGTCCCTGCACAAGGATGACACACAAATTCATGAAGTGTTCTATATTTTT,null,7414b474b3d15f0375a08d8eb8906fd1
15644574,URS0000EEB79E,2019-08-29T05:46:56.646Z,rnacen,77D79E0AE2B07F25,4078,null,TGGGACAAAGCAGGATCGCTGCTCAGCCCCAGCCCGGCAGAGCGGCCCAGGAGCCCCGGGTGACTGCCCCGGACTTCAGGGCCCGGCTTCGCTTTCCCCCTGCAGGACACTCCAGGTACTGCATGGCCCCGGGCCCCAAGGGCAGGGAGCGAGACCCTGGAGTGAAGACCCCCCGGCCCCGCCAGGTGTGATGGGGGGGCCCTGGGGATAGGGTCCCCGAAGCAAAGCTGAGACTCAGAGGTGGGCTGCTGAGCCACGCCGTCCGCAGGGGCGGTGGCTACAGGTAGTTGTCTTCTGCGGCGTCGCTCCCGCCCCTGGAGCCCCGGGGCTCATCCTCGTCGTCGTCGTCGTCATCGGGGGCCAGCGCCTCGATGTCGTGCGTGATGTCAGCCAGCAGCCGGTGGAAGGCCTGTGCCTCGGGCCGCTGCGCCGCGCCATCGTAGCTGCGCACTGCGGAGGCCGACGTGGAGCTCATGCTGCGCTTGATGCGGCCGAAGCTGTCAGTGAGGATGCCGCCCTCGCGGATGCCCACGCCCTCCAGGAAGCCCTCGAAGTAGCCGATGTCCTGGTCCGGGATGAAGGGCCGCATCCCTGCGAGCCGAGGGGCATGCCTCCAGGCTTTTGCTTGTGCCATTCCCACCACCGGCCCCAGCTCACCGTGACCATGTAATCCTGTAACTGCTCCAAGCCCAGACTGCTGTGGTCGCTGCCGCTGCAGTGGGAGCCATGGAAAGAAGGTGTGGACGTGCCGCTGTAACATGCTTCAAAGGTGTCCTGGGAGCCATTACTGCAGGCACGGG